---
## 1. Score NSE
### 1.1 Importaciones y rutas

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import glob
import os
import osmnx as ox
import ee
import geemap
import requests
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
import ftfy
import warnings
warnings.filterwarnings('ignore')

RUTA_ACTUAL=os.getcwd()
RUTA_SHP    =  RUTA_ACTUAL + r"\DATA\889463807469_s"
RUTA_CENSO  = RUTA_ACTUAL + r"\DATA\ageb"
RUTA_C1     = RUTA_ACTUAL + r"\DATA\C1_IngresosAltos"
RUTA_C2     = RUTA_ACTUAL + r"\DATA\C2_IngresosMedios"

In [2]:
print(RUTA_CENSO)

c:\Users\xboxn\Documents\QgisProy\DATA\ageb


### 1.2 Clase auxiliar para carga de negocios

In [3]:
class AnalisisNSE:

    def __init__(self):
        self.negocios = None
        self.shp = None
        self.negocios_cp = None

    def cargar_negocios(self, carpeta):
        archivos = glob.glob(carpeta + "\\**\\*.csv", recursive=True)
        dfs = []
        for archivo in archivos:
            df = pd.read_csv(archivo, encoding='latin-1')
            dfs.append(df)
        self.negocios = pd.concat(dfs, ignore_index=True)
        print(f"Negocios cargados: {self.negocios.shape[0]} registros")
        return self


### 1.3 Carga y limpieza del Censo 2020

In [4]:
columnas_nse = ['ENTIDAD', 'NOM_ENT', 'MUN', 'NOM_MUN', 'LOC', 'AGEB',
                'GRAPROES', 'P18YM_PB', 'VPH_AUTOM',
                'VPH_PC', 'VPH_INTER', 'PDER_IMSS', 'POCUPADA', 'POBTOT']

archivos_censo = glob.glob(RUTA_CENSO +r"/*.csv")
print(archivos_censo)
dfs_censo = []
for archivo in archivos_censo:
    
    df = pd.read_csv(archivo, encoding='latin-1', nrows=0)
    df.columns = [col.replace('ï»¿', '') for col in df.columns]
    df = pd.read_csv(archivo, encoding='latin-1', usecols=lambda c: c.replace('ï»¿', '') in columnas_nse)
    df.columns = [col.replace('ï»¿', '') for col in df.columns]
    dfs_censo.append(df)

df_censo_mexico = pd.concat(dfs_censo, ignore_index=True)
print(df_censo_mexico.shape)
print(df_censo_mexico.head())

['c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\ageb\\conjunto_de_datos_ageb_urbana_01_cpv2020.csv', 'c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\ageb\\conjunto_de_datos_ageb_urbana_02_cpv2020.csv', 'c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\ageb\\conjunto_de_datos_ageb_urbana_03_cpv2020.csv', 'c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\ageb\\conjunto_de_datos_ageb_urbana_04_cpv2020.csv', 'c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\ageb\\conjunto_de_datos_ageb_urbana_05_cpv2020.csv', 'c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\ageb\\conjunto_de_datos_ageb_urbana_06_cpv2020.csv', 'c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\ageb\\conjunto_de_datos_ageb_urbana_07_cpv2020.csv', 'c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\ageb\\conjunto_de_datos_ageb_urbana_08_cpv2020.csv', 'c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\ageb\\conjunto_de_datos_ageb_urbana_09_cpv2020.csv', 'c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\ageb\\conjunto_de_datos_ageb_urbana_10_cpv2020.csv', 'c:\\User

### 1.4 Construcción del score NSE

In [5]:
cols_numericas = ['GRAPROES', 'P18YM_PB', 'VPH_AUTOM',
                  'VPH_PC', 'VPH_INTER', 'PDER_IMSS', 'POCUPADA', 'POBTOT']

cols_suma = ['P18YM_PB', 'VPH_AUTOM', 'VPH_PC', 
             'VPH_INTER', 'PDER_IMSS', 'POCUPADA', 'POBTOT']

for col in cols_numericas:
    df_censo_mexico[col] = pd.to_numeric(df_censo_mexico[col], errors='coerce')

# Filtrar totales
df_ageb = df_censo_mexico[
    (df_censo_mexico['AGEB'] != '0000') &
    (df_censo_mexico['MUN']  != 0)
].copy()

# Sumar conteos por AGEB
agg_suma = df_censo_mexico.groupby(
    ['ENTIDAD', 'NOM_ENT', 'MUN', 'NOM_MUN', 'LOC', 'AGEB']
)[cols_suma].sum()

# Promedio ponderado de escolaridad por población
df_ageb['graproes_pond'] = df_ageb['GRAPROES'] * df_ageb['POBTOT']
agg_escolaridad = (
    df_ageb.groupby(['ENTIDAD', 'NOM_ENT', 'MUN', 'NOM_MUN', 'AGEB'])
    .apply(lambda x: x['graproes_pond'].sum() / x['POBTOT'].sum(), include_groups=False)
)
agg_escolaridad.name = 'GRAPROES'

df_ageb = agg_suma.join(agg_escolaridad).reset_index()

print(f"AGEBs únicas: {df_ageb.shape[0]:,}")
print("\nNulos por columna:")
print(df_ageb[cols_numericas].isna().sum())
print("\nEjemplo de AGEBs limpias:")
print(df_ageb[['NOM_MUN', 'AGEB', 'POBTOT', 'GRAPROES', 'POCUPADA']].head(5))

AGEBs únicas: 68,674

Nulos por columna:
GRAPROES     9713
P18YM_PB        0
VPH_AUTOM       0
VPH_PC          0
VPH_INTER       0
PDER_IMSS       0
POCUPADA        0
POBTOT          0
dtype: int64

Ejemplo de AGEBs limpias:
                              NOM_MUN  AGEB   POBTOT   GRAPROES  POCUPADA
0  Total de la entidad Aguascalientes  0000  1425607        NaN  692983.0
1                      Aguascalientes  0000   948990        NaN  476502.0
2                      Aguascalientes  0000   863893        NaN  436364.0
3                      Aguascalientes  0017     4474   9.228391    2162.0
4                      Aguascalientes  006A     2822  14.021300    1310.0


In [6]:
#Construir ratios 

scaler = MinMaxScaler()

pop = df_ageb['POBTOT'].replace(0, np.nan)   
ocu = df_ageb['POCUPADA'].replace(0, np.nan)  

df_ageb['r_escolaridad'] = df_ageb['GRAPROES']
 
df_ageb['r_empleo_formal'] = df_ageb['PDER_IMSS'] / ocu
 
df_ageb['r_motorizacion'] = df_ageb['VPH_AUTOM'] / pop * 100
 
df_ageb['r_tecnologia'] = (df_ageb['VPH_PC'] + df_ageb['VPH_INTER']) / pop * 100
 
df_ageb['r_ocupacion'] = df_ageb['POCUPADA'] / pop
 
ratios = ['r_escolaridad', 'r_empleo_formal', 'r_motorizacion', 'r_tecnologia', 'r_ocupacion']
 
df_ageb['r_empleo_formal'] = df_ageb['r_empleo_formal'].clip(upper=1.0)
 
for ratio in ['r_escolaridad', 'r_motorizacion', 'r_tecnologia', 'r_ocupacion']:
    techo = df_ageb[ratio].quantile(0.99)
    df_ageb[ratio] = df_ageb[ratio].clip(upper=techo)
 
df_ageb[ratios] = scaler.fit_transform(df_ageb[ratios].fillna(0))
 
print("Distribución después de normalizar:")
print(df_ageb[ratios].describe().round(3))

Distribución después de normalizar:
       r_escolaridad  r_empleo_formal  r_motorizacion  r_tecnologia  \
count      68674.000        68674.000       68674.000     68674.000   
mean           0.520            0.594           0.361         0.319   
std            0.279            0.378           0.245         0.253   
min            0.000            0.000           0.000         0.000   
25%            0.435            0.227           0.190         0.112   
50%            0.587            0.674           0.336         0.287   
75%            0.696            1.000           0.510         0.474   
max            1.000            1.000           1.000         1.000   

       r_ocupacion  
count    68674.000  
mean         0.663  
std          0.220  
min          0.000  
25%          0.628  
50%          0.725  
75%          0.786  
max          1.000  


In [7]:
#Calcular score final 
 
PESOS = {
    'r_escolaridad':   0.25,
    'r_empleo_formal': 0.25,
    'r_motorizacion':  0.20,
    'r_tecnologia':    0.15,
    'r_ocupacion':     0.15,
}
 

df_ageb['score_nse'] = sum(
    df_ageb[ratio] * peso
    for ratio, peso in PESOS.items()
)
 
df_ageb['score_nse'] = df_ageb['score_nse'] * 100
 
bins   = [0,  20,  40,  60,  80, 100]
labels = ['E/D', 'D+', 'C', 'C+', 'A/B']
 
df_ageb['nivel_nse'] = pd.cut(
    df_ageb['score_nse'],
    bins=bins,
    labels=labels,
    include_lowest=True
)
 
print("Distribución de niveles NSE en México:")
print(df_ageb['nivel_nse'].value_counts().sort_index())
 
print("\nEjemplo de AGEBs con su score:")
print(df_ageb[['NOM_MUN', 'AGEB', 'score_nse', 'nivel_nse']].head(10).to_string(index=False))
 
print("\nEstadísticas del score final:")
print(df_ageb['score_nse'].describe().round(2))

Distribución de niveles NSE en México:
nivel_nse
E/D     8138
D+     13900
C      20338
C+     20346
A/B     5952
Name: count, dtype: int64

Ejemplo de AGEBs con su score:
                           NOM_MUN AGEB  score_nse nivel_nse
Total de la entidad Aguascalientes 0000  52.877136         C
                    Aguascalientes 0000  54.826629         C
                    Aguascalientes 0000  55.371765         C
                    Aguascalientes 0017  58.383386         C
                    Aguascalientes 006A  87.788552       A/B
                    Aguascalientes 0106  93.305023       A/B
                    Aguascalientes 0163  77.708783        C+
                    Aguascalientes 0182  91.328582       A/B
                    Aguascalientes 0229  92.014403       A/B
                    Aguascalientes 0233  78.605098        C+

Estadísticas del score final:
count    68674.00
mean        49.82
std         23.54
min          0.00
25%         34.09
50%         52.50
75%         66.96


### 1.5 Merge con shapefiles

In [8]:
archivos_shp = glob.glob(RUTA_SHP+ r"/*.shp", recursive=True)
print(f"Shapefiles encontrados: {len(archivos_shp)}")
 
gdfs = []
for shp in archivos_shp:
    g = gpd.read_file(shp)
    g = g.to_crs(epsg=6372)
    gdfs.append(g)
 
ageb_shp = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True))
print(f"Polígonos cargados: {ageb_shp.shape[0]:,}")
print(f"Columnas del shapefile: {list(ageb_shp.columns)}")


Shapefiles encontrados: 32
Polígonos cargados: 63,982
Columnas del shapefile: ['CVEGEO', 'CVE_ENT', 'CVE_MUN', 'CVE_LOC', 'CVE_AGEB', 'geometry']


In [9]:
df_ageb['CVEGEO'] = (
    df_ageb['ENTIDAD'].astype(str).str.zfill(2) +
    df_ageb['MUN'].astype(str).str.zfill(3) +
    df_ageb['LOC'].astype(str).str.zfill(4) +
    df_ageb['AGEB'].astype(str).str.zfill(4)
)
print(f"Longitud CVEGEO censo: {df_ageb['CVEGEO'].str.len().value_counts()}")

cols_score = ['CVEGEO', 'NOM_ENT', 'NOM_MUN', 'POBTOT', 'score_nse', 'nivel_nse'] + ratios
gdf = ageb_shp.merge(df_ageb[cols_score], on='CVEGEO', how='left')

matched = gdf['score_nse'].notna().sum()
print(f"Polígonos con score: {matched:,} / {gdf.shape[0]:,}")
print(f"Sin match: {gdf.shape[0] - matched:,}")



Longitud CVEGEO censo: CVEGEO
13    68674
Name: count, dtype: int64
Polígonos con score: 60,777 / 63,983
Sin match: 3,206


### 1.6 Agregar negocios DENUE y clasificación

In [10]:
analisis_c1 = AnalisisNSE()
analisis_c1.cargar_negocios(RUTA_C1)
analisis_c1.negocios['categoria'] = 'C1'

analisis_c2 = AnalisisNSE()
analisis_c2.cargar_negocios(RUTA_C2)
analisis_c2.negocios['categoria'] = 'C2'

negocios = pd.concat(
    [analisis_c1.negocios, analisis_c2.negocios],
    ignore_index=True
)
print(negocios['categoria'].value_counts())


Negocios cargados: 5863 registros
Negocios cargados: 34548 registros
categoria
C2    34548
C1     5863
Name: count, dtype: int64


In [11]:
negocios['CVEGEO'] = (
    negocios['Clave entidad'].astype(str).str.zfill(2) +
    negocios['Clave municipio'].astype(str).str.zfill(3) +
    negocios['Clave localidad'].astype(str).str.zfill(4) +
    negocios['Área geoestadística básica '].astype(str).str.strip().str.zfill(4)
)
print(f"Longitud CVEGEO negocios: {negocios['CVEGEO'].str.len().value_counts()}")

c1_por_ageb = negocios[negocios['categoria'] == 'C1'].groupby('CVEGEO').size().reset_index(name='n_c1')
c2_por_ageb = negocios[negocios['categoria'] == 'C2'].groupby('CVEGEO').size().reset_index(name='n_c2')


Longitud CVEGEO negocios: CVEGEO
14    15801
16    13789
15     6860
17     3739
18      220
19        2
Name: count, dtype: int64


In [12]:
gdf = gdf.merge(c1_por_ageb, on='CVEGEO', how='left')
gdf = gdf.merge(c2_por_ageb, on='CVEGEO', how='left')
gdf['n_c1'] = gdf['n_c1'].fillna(0).astype(int)
gdf['n_c2'] = gdf['n_c2'].fillna(0).astype(int)
gdf['tiene_c1'] = gdf['n_c1'] > 0
gdf['tiene_c2'] = gdf['n_c2'] > 0

print(f"AGEBs: {len(gdf):,}")


AGEBs: 63,983


In [13]:
gdf['clasificacion'] = 'NSE bajo'

gdf.loc[(gdf['score_nse'] >= 40) & (gdf['score_nse'] < 60) & ~gdf['tiene_c1'] & ~gdf['tiene_c2'], 'clasificacion'] = 'NSE medio + Sin negocios'
gdf.loc[(gdf['score_nse'] >= 40) & (gdf['score_nse'] < 60) & ~gdf['tiene_c1'] &  gdf['tiene_c2'], 'clasificacion'] = 'NSE medio + Negocios medios'
gdf.loc[(gdf['score_nse'] >= 40) & (gdf['score_nse'] < 60) &  gdf['tiene_c1'], 'clasificacion'] = 'NSE medio + Negocios premium'
gdf.loc[(gdf['score_nse'] >= 60) & ~gdf['tiene_c1'] & ~gdf['tiene_c2'], 'clasificacion'] = 'NSE alto + Sin negocios'
gdf.loc[(gdf['score_nse'] >= 60) & ~gdf['tiene_c1'] &  gdf['tiene_c2'], 'clasificacion'] = 'NSE alto + Negocios medios'
gdf.loc[(gdf['score_nse'] >= 60) &  gdf['tiene_c1'], 'clasificacion'] = 'NSE alto + Negocios premium'

print(gdf['clasificacion'].value_counts())

clasificacion
NSE alto + Sin negocios     26189
NSE bajo                    19571
NSE medio + Sin negocios    18223
Name: count, dtype: int64


---
## 2. Pipeline Satelital GEE
### 2.1 Inicialización y carga de AGEBs de CDMX

In [ ]:
import ee
import geemap
from sklearn.cluster import KMeans

ee.Authenticate()
ee.Initialize(project='modular-height-414314')

# Simplificar geometrias
gdf_zmvm = gdf[gdf['CVEGEO'].str[:2].isin(['09', '15'])][['CVEGEO', 'geometry']].copy()
gdf_zmvm = gdf_zmvm.to_crs('EPSG:4326')
gdf_zmvm['geometry'] = gdf_zmvm['geometry'].simplify(tolerance=0.001)

print(f"AGEBs: {len(gdf_zmvm)}")

# Convertir a FeatureCollection en memoria
agebs = geemap.gdf_to_ee(gdf_zmvm)
print(f"AGEBs ZMVM: {agebs.size().getInfo():,}")

### 2.2 Funciones de índices espectrales

In [ ]:
def ndvi_anual(year, agebs):
    """NDVI promedio anual por AGEB usando Sentinel-2 SR Harmonized.
    B8 = NIR, B4 = Rojo. NDVI = (B8-B4)/(B8+B4)"""
    return ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterDate(f'{year}-01-01', f'{year}-12-31') \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
        .filterBounds(agebs) \
        .map(lambda img: img.normalizedDifference(['B8', 'B4']).rename('NDVI')) \
        .median()

def ndbi_anual(year, agebs):
    """NDBI promedio anual por AGEB usando Sentinel-2 SR Harmonized.
    B11 = SWIR, B8 = NIR. NDBI = (B11-B8)/(B11+B8)
    Positivo = superficie construida. Negativo = vegetacion."""
    return ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterDate(f'{year}-01-01', f'{year}-12-31') \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
        .filterBounds(agebs) \
        .map(lambda img: img.normalizedDifference(['B11', 'B8']).rename('NDBI')) \
        .median()


In [1]:
gdf.columns

NameError: name 'gdf' is not defined

### 2.3 Cálculo de NDVI 2018-2024, NDBI 2024 y delta NDVI

In [ ]:
#Carga de variables satelitales

import glob

archivos = sorted(glob.glob('data/satelital_*.csv'))

if archivos:
    # Si ya hay, usa el más reciente
    df_satelital = pd.read_csv(archivos[-1])
    print(f"Usando datos satelitales: {archivos[-1]}")
else:
    # Por si es la primera vez
    import datetime
    year = datetime.datetime.now().year
    year_base = year - 6

    stats_actual = ndvi_anual(year, agebs).reduceRegions(
        collection=agebs, reducer=ee.Reducer.mean(), scale=100
    ).map(lambda f: f.set(f'ndvi_{year}', f.get('mean')))

    stats_base = ndvi_anual(year_base, agebs).reduceRegions(
        collection=agebs, reducer=ee.Reducer.mean(), scale=100
    ).map(lambda f: f.set(f'ndvi_{year_base}', f.get('mean')))

    stats_ndbi = ndbi_anual(year, agebs).reduceRegions(
        collection=agebs, reducer=ee.Reducer.mean(), scale=100
    ).map(lambda f: f.set(f'ndbi_{year}', f.get('mean')))

    df_satelital = geemap.ee_to_df(stats_actual.select(
        ['CVEGEO', f'ndvi_{year}']
    )).merge(
        geemap.ee_to_df(stats_base.select(['CVEGEO', f'ndvi_{year_base}'])),
        on='CVEGEO'
    ).merge(
        geemap.ee_to_df(stats_ndbi.select(['CVEGEO', f'ndbi_{year}'])),
        on='CVEGEO'
    )

    df_satelital['delta_ndvi'] = df_satelital[f'ndvi_{year}'] - df_satelital[f'ndvi_{year_base}']
    df_satelital.rename(columns={f'ndvi_{year}': 'ndvi_2024', f'ndbi_{year}': 'ndbi_2024'}, inplace=True)

    os.makedirs('data', exist_ok=True)
    df_satelital.to_csv(f'data/satelital_{year}.csv', index=False)

df_satelital = df_satelital.merge(
    gdf[['CVEGEO', 'score_nse', 'nivel_nse', 'clasificacion']],
    on='CVEGEO', how='left'
)

print(f"Shape: {df_satelital.shape}")
print(df_satelital.head())


Shape: (6751, 8)
          CVEGEO  ndvi_2024  ndvi_2020  ndbi_2024  delta_ndvi  score_nse  \
0  0901000011716   0.187073   0.184705   0.028288    0.002368  57.640460   
1  0901000012150   0.130572   0.121877   0.037729    0.008696  78.746480   
2  0901000011133   0.363880   0.345051  -0.079746    0.018829  87.434444   
3  0901000011307   0.295527   0.301144  -0.037255   -0.005617  79.264413   
4  0901000010281   0.302393   0.325711  -0.023781   -0.023317  77.326333   

  nivel_nse                clasificacion  
0         C     NSE medio + Sin negocios  
1        C+      NSE alto + Sin negocios  
2       A/B  NSE alto + Negocios premium  
3        C+   NSE alto + Negocios medios  
4        C+      NSE alto + Sin negocios  


### 2.4 Clustering satelital (K-means k=4)

In [ ]:
X = df_satelital[['ndvi_2024', 'ndbi_2024', 'delta_ndvi']].dropna()

km = KMeans(n_clusters=4, random_state=42)
km.fit(X)
df_satelital['cluster'] = km.labels_

print("Distribucion de clusters:")
print(df_satelital['cluster'].value_counts())

print("\nPerfil promedio por cluster:")
print(df_satelital.groupby('cluster')[['ndvi_2024', 'ndbi_2024', 'delta_ndvi']].mean())

# Interpretacion de clusters (validada con Inspector de GEE):
#   Cluster 0 (Rojo)   - Urbano denso estable
#   Cluster 1 (Azul)   - Verde natural/periurbano
#   Cluster 2 (Verde)  - Urbano mixto degradandose
#   Cluster 3 (Morado) - Urbano denso degradandose


Distribucion de clusters:
cluster
2    2962
1    2170
0    1246
3     373
Name: count, dtype: int64

Perfil promedio por cluster:
         ndvi_2024  ndbi_2024  delta_ndvi
cluster                                  
0         0.260349   0.013064   -0.003557
1         0.183485   0.071709   -0.012293
2         0.104696   0.064939   -0.002608
3         0.406486  -0.056428   -0.005694


### 2.5 Merge al gdf principal

In [ ]:
gdf = gdf[gdf['CVE_ENT'].isin(['09', '15'])].copy()
gdf = gdf.merge(df_satelital[['CVEGEO', 'delta_ndvi', 'ndbi_2024', 'ndvi_2024', 'cluster']], on='CVEGEO', how='left')
print(f"Columnas finales: {gdf.columns.tolist()}")
print(f"AGEBs totales: {len(gdf):,}")


Columnas finales: ['CVEGEO', 'CVE_ENT', 'CVE_MUN', 'CVE_LOC', 'CVE_AGEB', 'geometry', 'NOM_ENT', 'NOM_MUN', 'POBTOT', 'score_nse', 'nivel_nse', 'r_escolaridad', 'r_empleo_formal', 'r_motorizacion', 'r_tecnologia', 'r_ocupacion', 'n_c1', 'n_c2', 'tiene_c1', 'tiene_c2', 'clasificacion', 'delta_ndvi', 'ndbi_2024', 'ndvi_2024', 'cluster']
AGEBs totales: 6,751


## 3. Pipeline Satelital GEE

In [ ]:
import osmnx as ox

landuse = ox.features_from_place(
    "Ciudad de México, México",
    tags={"landuse": ["residential", "commercial", "industrial", "grass", "forest"]}
)

print(landuse.shape)
print(landuse['landuse'].value_counts())

(6246, 150)
landuse
grass          3017
residential    2268
commercial      466
industrial      323
forest          172
Name: count, dtype: int64


In [ ]:
landuse_clean = landuse[['geometry', 'landuse']].copy()

landuse_clean = landuse_clean[
    landuse_clean.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])
].reset_index(drop=True)

print(landuse_clean.shape)
print(landuse_clean.crs)

(6233, 2)
epsg:4326


In [ ]:
landuse_clean = landuse_clean.to_crs(gdf.crs)

In [ ]:
import geopandas as gpd

j_landuse = gpd.sjoin(
    gdf[['geometry', 'CVEGEO']], 
    landuse_clean,
    how='left',
    predicate='intersects'
)

landuse_dominante = (
    j_landuse.groupby('CVEGEO')['landuse']
    .agg(lambda x: x.dropna().value_counts().index[0] 
         if len(x.dropna()) > 0 
         else 'sin_dato')
    .reset_index()
    .rename(columns={'landuse': 'landuse_dom'})
)

print(landuse_dominante['landuse_dom'].value_counts())

landuse_dom
sin_dato       5685
residential     479
grass           348
commercial       94
industrial       77
forest           68
Name: count, dtype: int64


In [ ]:
gdf = gdf.merge(landuse_dominante, on='CVEGEO', how='left')
gdf['landuse_dom'] = gdf['landuse_dom'].fillna('sin_dato')
print(gdf['landuse_dom'].value_counts())

landuse_dom
sin_dato       5685
residential     479
grass           348
commercial       94
industrial       77
forest           68
Name: count, dtype: int64


## Redes viales (Accesibilidad)

In [ ]:
import networkx as nx

In [ ]:
G = ox.graph_from_place("Ciudad de México, México", network_type="drive")

G = ox.add_edge_speeds(G)
G = ox.add_edge_travel_times(G)

print(f"Nodos: {len(G.nodes):,}")
print(f"Aristas: {len(G.edges):,}")

Nodos: 125,564
Aristas: 295,097


In [ ]:
G

In [ ]:
#Accesibilidad
gdf_g = gdf[gdf['CVE_ENT'].isin(['09', '15'])].copy()
gdf_g = gdf_g.to_crs('EPSG:4326')
gdf_g['centroide'] = gdf_g.geometry.centroid

# Nodo más cercano por AGEB
gdf_g['nodo'] = ox.nearest_nodes(
    G,
    gdf_g['centroide'].x,
    gdf_g['centroide'].y
)

print(f"AGEBs con nodo asignado: {gdf_g['nodo'].notna().sum():,}")

AGEBs con nodo asignado: 6,751


In [ ]:
TIEMPO = 15 * 60 

def accesibilidad(nodo, G, gdf_g, tiempo):
    try:
        nodos_alcanzables = nx.single_source_dijkstra_path_length(
            G, nodo, cutoff=tiempo, weight='travel_time'
        )
        
        agebs_dentro = gdf_g[gdf_g['nodo'].isin(nodos_alcanzables.keys())]
        
        return {
            'n_agebs': len(agebs_dentro),
            'n_nse_alto': (agebs_dentro['nivel_nse'] == 'A/B').sum(),
            'n_premium': (agebs_dentro['clasificacion'] == 'NSE alto + Negocios premium').sum(),
        }
    except:
        return {'n_agebs': 0, 'n_nse_alto': 0, 'n_premium': 0}

resultados = gdf_g['nodo'].apply(lambda n: accesibilidad(n, G, gdf_g, TIEMPO))
df_acc = pd.DataFrame(resultados.tolist())

gdf_g = pd.concat([gdf_g.reset_index(drop=True), df_acc], axis=1)

print(gdf_g[['CVEGEO', 'n_agebs', 'n_nse_alto', 'n_premium']].head(10))

          CVEGEO  n_agebs  n_nse_alto  n_premium
0  0901000011716      212          75          6
1  0901000012150     1089         459         28
2  0901000011133      984         435         25
3  0901000011307      675         333         19
4  0901000010281      676         233         24
5  0901000012199      651         323         23
6  0901000012269      607         326         26
7  0901000012273      311         102         10
8  0901000011330      208          69          6
9  090100001181A      379         207         17


In [ ]:
scaler = MinMaxScaler(feature_range=(0, 100))

gdf_g['score_accesibilidad'] = scaler.fit_transform(gdf_g[['n_nse_alto']])

print(gdf_g[['CVEGEO', 'n_agebs', 'n_nse_alto', 'n_premium', 'score_accesibilidad']].head(10))
print(f"\nEstadísticas:")
print(gdf_g['score_accesibilidad'].describe().round(2))

          CVEGEO  n_agebs  n_nse_alto  n_premium  score_accesibilidad
0  0901000011716      212          75          6            12.295082
1  0901000012150     1089         459         28            75.245902
2  0901000011133      984         435         25            71.311475
3  0901000011307      675         333         19            54.590164
4  0901000010281      676         233         24            38.196721
5  0901000012199      651         323         23            52.950820
6  0901000012269      607         326         26            53.442623
7  0901000012273      311         102         10            16.721311
8  0901000011330      208          69          6            11.311475
9  090100001181A      379         207         17            33.934426

Estadísticas:
count    6751.00
mean       23.67
std        24.07
min         0.00
25%         9.02
50%        15.90
75%        41.64
max       100.00
Name: score_accesibilidad, dtype: float64


In [ ]:
import glob

archivos_opp = glob.glob(r"C:\Users\samdc\Downloads\Prácticas Profesionales\**\Zonas_Potenciales_Negocio.shp", recursive=True)

if archivos_opp:
    RUTA_OPP = archivos_opp[0]
    print(f"Usando: {RUTA_OPP}")
else:
    print("No se encontró el shapefile — corre model.ipynb primero")

score_opp = gpd.read_file(RUTA_OPP)
score_opp = score_opp.to_crs(gdf_g.crs)

# Centroide de cada AGEB
gdf_centroides = gdf_g[['CVEGEO', 'geometry']].copy()
gdf_centroides['geometry'] = gdf_g.geometry.centroid

# Cada AGEB recibe el score_opp de la zona en que cae su centroide
joined = gpd.sjoin(
    gdf_centroides,
    score_opp[['score_opp', 'geometry']],
    how='left',
    predicate='within'
)

# Merge al GDF principal
opp_por_ageb = joined[['CVEGEO', 'score_opp']].drop_duplicates('CVEGEO')
gdf_g = gdf_g.merge(opp_por_ageb, on='CVEGEO', how='left')
gdf_g['score_opp'] = gdf_g['score_opp'].fillna(0)

print(gdf_g['score_opp'].describe().round(2))
print(f"AGEBs con score_opp > 0: {(gdf_g['score_opp'] > 0).sum()}")

Usando: C:\Users\samdc\Downloads\Prácticas Profesionales\modelo_negocios\Zonas_Potenciales_Negocio.shp
count    6751.00
mean       -0.05
std         0.06
min        -0.30
25%        -0.10
50%        -0.10
75%         0.00
max         0.00
Name: score_opp, dtype: float64
AGEBs con score_opp > 0: 0


In [ ]:
score_opp = gpd.read_file(RUTA_OPP)
print(score_opp['score_opp'].describe().round(4))
print(f"Zonas positivas: {(score_opp['score_opp'] > 0).sum()}")
print(f"Zonas negativas: {(score_opp['score_opp'] < 0).sum()}")

count    39.0000
mean      0.6949
std       0.7725
min      -0.3000
25%       0.1000
50%       0.5000
75%       1.3000
max       2.3000
Name: score_opp, dtype: float64
Zonas positivas: 31
Zonas negativas: 8


In [ ]:
print(gdf_g['score_opp'].describe().round(4))
print(f"AGEBs con score_opp > 0: {(gdf_g['score_opp'] > 0).sum()}")
print(f"AGEBs con score_opp < 0: {(gdf_g['score_opp'] < 0).sum()}")
print(f"AGEBs con score_opp NaN: {gdf_g['score_opp'].isna().sum()}")

count    6751.0000
mean       -0.0544
std         0.0567
min        -0.3000
25%        -0.1000
50%        -0.1000
75%         0.0000
max         0.0000
Name: score_opp, dtype: float64
AGEBs con score_opp > 0: 0
AGEBs con score_opp < 0: 3509
AGEBs con score_opp NaN: 0


In [ ]:
# Score final combinado (Prototipo)

PESOS_FINAL = {
    'score_nse':           0.30,
    'score_accesibilidad': 0.25,
    'score_opp':           0.30,
    'ndbi_2024':           0.10,
    'delta_ndvi':          0.05
}

gdf_g['ndbi_norm'] = MinMaxScaler(feature_range=(0,100)).fit_transform(
    gdf_g[['ndbi_2024']].fillna(0)
)

gdf_g['delta_ndvi_norm'] = MinMaxScaler(feature_range=(0,100)).fit_transform(
    gdf_g[['delta_ndvi']].fillna(0)
)

gdf_g['score_final'] = (
    gdf_g['score_nse'].fillna(0) * PESOS_FINAL['score_nse'] +
    gdf_g['score_accesibilidad'].fillna(0)* PESOS_FINAL['score_accesibilidad'] +
    gdf_g['ndbi_norm'].fillna(0) * PESOS_FINAL['ndbi_2024'] +
    gdf_g['delta_ndvi_norm'].fillna(0) * PESOS_FINAL['delta_ndvi']
)

print(gdf_g[['CVEGEO', 'score_nse', 'score_accesibilidad', 'score_final']].head(10))
print(f"\nEstadísticas score final:")
print(gdf_g['score_final'].describe().round(2))

          CVEGEO  score_nse  score_accesibilidad  score_final
0  0901000011716  57.640460            12.295082    30.234968
1  0901000012150  78.746480            75.245902    52.612823
2  0901000011133  87.434444            71.311475    52.060885
3  0901000011307  79.264413            54.590164    45.827539
4  0901000010281  77.326333            38.196721    41.087914
5  0901000012199  67.978194            52.950820    43.692447
6  0901000012269  75.419361            53.442623    43.735411
7  0901000012273  83.228603            16.721311    37.684498
8  0901000011330  65.420395            11.311475    30.517782
9  090100001181A  66.941714            33.934426    38.200469

Estadísticas score final:
count    6751.00
mean       34.71
std         9.18
min         7.33
25%        28.21
50%        33.26
75%        39.89
max        61.01
Name: score_final, dtype: float64


In [ ]:
cols_finales = ['CVEGEO', 'score_nse', 'nivel_nse', 'clasificacion',
                'delta_ndvi', 'ndbi_2024', 'ndvi_2024', 'cluster', 
                'landuse_dom', 'score_accesibilidad', 'score_opp',
                'score_final', 'geometry']

gdf_export = gdf_g[cols_finales].copy()
gdf_export = gdf_export.to_crs('EPSG:4326')
gdf_export['geometry'] = gdf_export['geometry'].simplify(tolerance=0.001)
gdf_export['nivel_nse'] = gdf_export['nivel_nse'].astype(str)

fc_final = geemap.gdf_to_ee(gdf_export)

task = ee.batch.Export.table.toAsset(
    collection=fc_final,
    description='zmvm_final_proto',
    assetId='projects/ee-practicas-satelites/assets/zmvm_final_proto'
)
task.start()
print(task.status())

{'state': 'READY', 'description': 'zmvm_final_proto', 'priority': 100, 'creation_timestamp_ms': 1777592495594, 'update_timestamp_ms': 1777592495594, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'X4ZD73QDLZU4W5MTM4LJP4Z4', 'name': 'projects/ee-practicas-satelites/operations/X4ZD73QDLZU4W5MTM4LJP4Z4'}


var zmvm = ee.FeatureCollection('projects/ee-practicas-satelites/assets/zmvm_final_proto');

var imagen = zmvm.reduceToImage({
  properties: ['score_final'],
  reducer: ee.Reducer.mean()
}).reproject({
  crs: 'EPSG:4326',
  scale: 100
});

Map.centerObject(zmvm, 9);
Map.addLayer(imagen, {
  min: 7, 
  max: 61, 
  palette: ['d73027', 'fee08b', '1a9850']
}, 'Score Final Proto');